# AI Engineering Level 1 – Module 4
## Beginner Google Colab Workbook: Machine Learning Fundamentals with Scikit-Learn

**Labs 1–4 | Beginner level | Python**

### What you will learn
By completing this notebook, you will be able to:
1. Explain supervised learning and unsupervised learning.
2. Split a dataset into training and testing parts.
3. Scale numeric features correctly without data leakage.
4. Train a Logistic Regression model.
5. Read prediction probabilities and evaluate a classifier.

### How to use this notebook
- Run each code cell from top to bottom.
- Read the explanation before running the code.
- Change the small values suggested in the exercises.
- Do not worry if the output is slightly different across software versions.
- In Google Colab, choose **Runtime → Run all** to run the whole notebook.

**Important:** The Breast Cancer dataset is used only as an educational example. This notebook is not a medical diagnostic tool.


---
# 0. Setup – Prepare Your Python Environment

In this section, we import the tools used in the four labs.

- **NumPy** helps us work with arrays and numbers.
- **Scikit-Learn** provides datasets and machine-learning algorithms.


In [ ]:
# Install Scikit-Learn if it is not already available
!pip -q install scikit-learn

In [ ]:
# Import the libraries we will use
import numpy as np
import sklearn

print("NumPy version:", np.__version__)
print("Scikit-Learn version:", sklearn.__version__)
print("Setup completed!")

---
# LAB 1 – Supervised vs Unsupervised Learning

## 1. Objective
In this lab, you will:
- Understand the difference between supervised and unsupervised learning.
- Train a simple classification model.
- Use KMeans to group similar data points.

## 2. Concept

### Supervised learning
Supervised learning uses examples that already have the correct answers, called **labels**.

Example: An email dataset contains messages labeled `spam` or `not spam`. The model learns from those examples and predicts a label for a new email.

### Unsupervised learning
Unsupervised learning uses data without known answer labels. The algorithm tries to discover groups or patterns.

Example: A shop groups customers by similar purchasing behavior without having predefined customer-group labels.

### Key words
- **Feature (X):** information given to the model.
- **Label (y):** the correct answer in supervised learning.
- **Classification:** predicting a category.
- **Clustering:** grouping similar records.


## 3. Code – Part A: Supervised Classification

We will create a practice dataset. Each row has 10 numeric features and a class label. We will train Logistic Regression to predict the label.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Create 500 sample records with 10 features
X, y = make_classification(
    n_samples=500,
    n_features=10,
    n_informative=6,
    random_state=42
)

print("Feature data shape:", X.shape)
print("Label data shape:", y.shape)
print("First 5 labels:", y[:5])

### Understand the output
- `X.shape` shows the number of rows and features.
- `y.shape` shows how many labels we have.
- `y[:5]` displays the first five labels.

The value `random_state=42` makes the generated practice data repeatable.

In [ ]:
# Split the data: 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

**Why split the data?**

The model learns from the training data. We then use the test data to check how well it works on records it did not train on.

`stratify=y` helps keep the class proportions similar in the training and test sets.

In [ ]:
# Create and train the model
model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)

# Predict labels for the test data
predictions = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, predictions)
print("Test accuracy:", round(accuracy, 4))

### What does accuracy mean?

Accuracy is the percentage of test records that the model classified correctly.

For example, an accuracy of `0.90` means 90% of the test predictions were correct. Accuracy alone does not tell us every kind of mistake the model makes; we will study more metrics in Lab 4.

## 4. Code – Part B: Unsupervised Clustering

Now we will create data points arranged around four groups. We will use KMeans to discover four clusters.

Notice that we do not provide the labels to KMeans.

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

# Create 300 data points around 4 generated groups
X_groups, _ = make_blobs(
    n_samples=300,
    centers=4,
    random_state=42
)

# Ask KMeans to find 4 clusters
kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X_groups)

print("Number of cluster centers:", len(kmeans.cluster_centers_))
print("Cluster center shape:", kmeans.cluster_centers_.shape)
print("First 10 cluster assignments:", cluster_labels[:10])

### Understand the output
- `n_clusters=4` asks KMeans to create four groups.
- `fit_predict()` learns the groups and assigns a cluster number to each data point.
- `cluster_centers_` contains the center of each discovered group.
- Cluster IDs such as 0, 1, 2, and 3 are group identifiers, not known real-world labels.

In [ ]:
# Count how many data points were assigned to each cluster
for cluster_id in range(4):
    count = np.sum(cluster_labels == cluster_id)
    print(f"Cluster {cluster_id}: {count} records")

## 5. Expected Output – Lab 1

Your output should show:
- Feature shape: `(500, 10)`
- Label shape: `(500,)`
- Training records: `400`
- Testing records: `100`
- Four cluster centers with shape `(4, 2)`
- Four cluster counts whose total is `300`

Accuracy and individual cluster counts are calculated when you run the notebook.

## 6. Exercise – Lab 1
1. Change `n_samples=500` to `n_samples=1000`. What happens to the feature shape?
2. Change `test_size=0.2` to `test_size=0.3`. How many test records do you get?
3. Change `centers=4` and `n_clusters=4` to `3`. What is the new cluster-center shape?
4. In your own words, explain why Logistic Regression is supervised but KMeans is unsupervised.

## 7. Challenge – Lab 1
Try KMeans with `n_clusters=2`, `3`, and `5`. Record the cluster-center shape and cluster sizes. Explain why choosing the number of clusters can affect the result.

## 8. AI Application – Lab 1
A company may use supervised learning to classify customer-support tickets into known categories. It may use unsupervised learning to discover groups of similar tickets when categories are not yet known.


---
# LAB 2 – Train/Test Split and StandardScaler

## 1. Objective
In this lab, you will:
- Load a real Scikit-Learn practice dataset.
- Check its rows, columns, and class distribution.
- Split it into training and testing data.
- Standardize features using `StandardScaler`.

## 2. Concept

### Train/test split
We train a model on one part of the dataset and evaluate it on a separate part.

### Stratified split
A stratified split tries to preserve the class proportions in both subsets. This is useful when one class has more records than another.

### Feature scaling
Features may have very different numeric ranges. `StandardScaler` transforms each feature using:

`scaled value = (value - training mean) / training standard deviation`

After scaling, each training feature is approximately centered around 0 with a standard deviation around 1.

**Important – avoid data leakage:** Fit the scaler using training data only. Then use that same fitted scaler to transform test data.


## 3. Code – Load and Inspect the Dataset

In [ ]:
from sklearn.datasets import load_breast_cancer

# Load the built-in practice dataset
dataset = load_breast_cancer()

# Separate the input features and target labels
X = dataset.data
y = dataset.target

print("Dataset shape:", X.shape)
print("Feature count:", X.shape[1])
print("Class names:", dataset.target_names)
print("First 5 target values:", y[:5])

The dataset contains 569 records and 30 numeric features. The target has two classes. In this dataset, class `0` is malignant and class `1` is benign.

In [ ]:
# Count records in each class
unique_classes, class_counts = np.unique(y, return_counts=True)

for class_id, count in zip(unique_classes, class_counts):
    print(f"Class {class_id} ({dataset.target_names[class_id]}): {count} records")

## 4. Code – Split the Dataset

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)
print("Training label count:", len(y_train))
print("Testing label count:", len(y_test))

## 5. Code – Scale the Features

In [ ]:
from sklearn.preprocessing import StandardScaler

# Create the scaler
scaler = StandardScaler()

# Learn scaling values from training data and scale training data
X_train_scaled = scaler.fit_transform(X_train)

# Use the same training scaling values on test data
X_test_scaled = scaler.transform(X_test)

print("Original first feature mean:", round(X_train[:, 0].mean(), 2))
print("Scaled first feature mean:", round(X_train_scaled[:, 0].mean(), 4))
print("Scaled first feature standard deviation:",
      round(X_train_scaled[:, 0].std(), 4))

### Understand `fit_transform()` and `transform()`

- `fit_transform(X_train)` learns the training-set mean and standard deviation, then scales the training data.
- `transform(X_test)` uses the already-learned training statistics to scale the test data.
- Do not call `fit_transform()` separately on the test data. That would use information from the test set during preprocessing.

The training feature's scaled mean should be close to 0 and its standard deviation close to 1.

## 6. Expected Output – Lab 2

You should see:
- Dataset shape: `(569, 30)`
- Class names: `['malignant' 'benign']`
- Training records: `455`
- Testing records: `114`
- Scaled training feature mean: close to `0`
- Scaled training feature standard deviation: close to `1`

The test feature's mean and standard deviation do not have to be exactly 0 and 1, because the scaler uses training statistics.

## 7. Exercise – Lab 2
1. Change `test_size=0.2` to `0.25`. How many records are in the test set?
2. Print the number of records for each class in `y_train` and `y_test`.
3. Compare the first feature before and after scaling.
4. Explain why we fit the scaler only on training data.

## 8. Challenge – Lab 2
Create a second scaler and fit it on `X_test`. Compare its output with `X_test_scaled`. Explain why the first workflow is preferred for fair model evaluation.

## 9. AI Application – Lab 2
In an AI model that uses customer age, income, and transaction amount, feature scaling can help prevent large-range features from dominating algorithms that are sensitive to feature scale.


---
# LAB 3 – Logistic Regression and Prediction Probabilities

## 1. Objective
In this lab, you will:
- Train a Logistic Regression classifier.
- Generate class predictions.
- Generate probability predictions.
- Understand the model's learned coefficients.

## 2. Concept

Logistic Regression is a supervised algorithm commonly used for classification.

It calculates a score and converts that score into a probability using the **sigmoid function**:

`sigmoid(z) = 1 / (1 + exp(-z))`

The result is between 0 and 1.

- `predict()` returns a class label.
- `predict_proba()` returns a probability for each class.

For this dataset:
- Class `0` = malignant
- Class `1` = benign

Always check the model's class order using `model.classes_`.


## 3. Code – Prepare Data and Train the Model

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Load data
dataset = load_breast_cancer()
X = dataset.data
y = dataset.target

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scale data without data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Create and train Logistic Regression
model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

print("Model training completed!")
print("Model class order:", model.classes_)

### What is `C`?

`C` controls the strength of regularization in Scikit-Learn Logistic Regression:
- Smaller `C` means stronger regularization.
- Larger `C` means weaker regularization.

`max_iter=1000` allows the optimizer more iterations to find a solution.

## 4. Code – Predict Labels and Probabilities

In [ ]:
# Predict the first five test records
predicted_labels = model.predict(X_test_scaled[:5])

# Get probabilities for the first five test records
predicted_probabilities = model.predict_proba(X_test_scaled[:5])

for i in range(5):
    print("Record:", i + 1)
    print("Actual label:", y_test[i])
    print("Predicted label:", predicted_labels[i])
    print("Probability of malignant (class 0):",
          round(predicted_probabilities[i, 0], 4))
    print("Probability of benign (class 1):",
          round(predicted_probabilities[i, 1], 4))
    print("-" * 35)

### Understand the probabilities

Each row of `predict_proba()` contains two probabilities. They should add up to approximately 1.

For example, `[0.10, 0.90]` means the model estimates:
- 10% probability for class 0 (malignant)
- 90% probability for class 1 (benign)

This is a model estimate, not a guarantee.

## 5. Code – Inspect Model Coefficients

In [ ]:
print("Coefficient array shape:", model.coef_.shape)

# Show the three largest coefficient magnitudes
absolute_coefficients = np.abs(model.coef_[0])
top_three = np.sort(absolute_coefficients)[::-1][:3]

print("Top 3 absolute coefficient values:", top_three.round(4))

A coefficient describes how a feature contributes to the model's decision, considering the other features in the model. The absolute values show magnitude, but do not by themselves explain causation or feature importance in every context.

## 6. Expected Output – Lab 3

You should see:
- A message confirming model training.
- Model class order: `[0 1]`
- Five actual labels and five predicted labels.
- Two probabilities for each record, adding to approximately 1.
- Coefficient array shape: `(1, 30)`.
- Three numeric coefficient magnitudes.

Your predicted labels and probability values are calculated when you run the notebook.

## 7. Exercise – Lab 3
1. Print `model.classes_`. What does the order mean?
2. Check whether each pair of probabilities adds to 1.
3. Change `C=1.0` to `C=0.1`. Compare the coefficients.
4. Change `C=1.0` to `C=10`. Compare the coefficients.
5. Explain the difference between `predict()` and `predict_proba()`.

## 8. Challenge – Lab 3
Use a probability threshold of `0.60` for class 1. Create your own predicted labels from the class-1 probabilities and compare them with the default predictions.

Hint:
`custom_predictions = (model.predict_proba(X_test_scaled)[:, 1] >= 0.60).astype(int)`

How does changing the threshold change the number of records predicted as class 1?

## 9. AI Application – Lab 3
A model can estimate the probability of a customer canceling a service. A company may use those estimates to decide which cases need further review. The threshold should be selected based on the costs of different mistakes.


---
# LAB 4 – Evaluate a Classification Model

## 1. Objective
In this lab, you will:
- Evaluate a trained classifier.
- Read a confusion matrix.
- Understand precision, recall, and F1-score.
- Calculate ROC-AUC.

## 2. Concept

### Confusion matrix
A confusion matrix compares actual labels with predicted labels.

For a binary classification problem, when class 1 is treated as the positive class:

| Term | Meaning |
|---|---|
| True Negative (TN) | Actual 0, predicted 0 |
| False Positive (FP) | Actual 0, predicted 1 |
| False Negative (FN) | Actual 1, predicted 0 |
| True Positive (TP) | Actual 1, predicted 1 |

### Metrics
- **Precision:** Of all records predicted positive, how many were truly positive?
- **Recall:** Of all truly positive records, how many did the model find?
- **F1-score:** A combined measure of precision and recall.
- **ROC-AUC:** Measures how well the model ranks positive examples above negative examples across thresholds.

Formulas:

- Precision = TP / (TP + FP)
- Recall = TP / (TP + FN)
- F1 = 2 × (Precision × Recall) / (Precision + Recall)


## 3. Code – Train and Predict

We repeat the preparation steps so this lab can run independently if needed.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

dataset = load_breast_cancer()
X = dataset.data
y = dataset.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_prob_class1 = model.predict_proba(X_test_scaled)[:, 1]

print("Predictions created:", len(y_pred))

## 4. Code – Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()

print("True Negative (TN):", tn)
print("False Positive (FP):", fp)
print("False Negative (FN):", fn)
print("True Positive (TP):", tp)

The matrix uses class order `[0, 1]`. In this lab, class 1 is treated as positive for the TN/FP/FN/TP names.

- A false positive is an actual class 0 record predicted as class 1.
- A false negative is an actual class 1 record predicted as class 0.

Because the dataset's class 0 is malignant and class 1 is benign, always state the class convention when discussing errors.

## 5. Code – Classification Report

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(
    y_test,
    y_pred,
    target_names=["malignant", "benign"]
)

print(report)

The report gives precision, recall, F1-score, and support for each class.

**Support** is the number of actual test records belonging to that class.

## 6. Code – ROC-AUC

In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test, y_prob_class1)
print("ROC-AUC score:", round(auc, 4))

ROC-AUC is generally between 0 and 1:
- 1.0 indicates perfect ranking on the evaluated data.
- 0.5 indicates ranking similar to random guessing.
- A higher score usually indicates better ranking discrimination.

ROC-AUC does not select the best threshold for your application. Threshold choice depends on the costs of false positives and false negatives.

## 7. Expected Output – Lab 4

Your output should include:
- A 2 × 2 confusion matrix.
- TN, FP, FN, and TP counts.
- A classification report for malignant and benign classes.
- A ROC-AUC score between 0 and 1.

The four confusion-matrix counts should add up to 114 test records for this split. Exact metric values are produced at runtime.

## 8. Exercise – Lab 4
1. Which confusion-matrix cell represents false positives?
2. Which cell represents false negatives?
3. Calculate precision for class 1 using the matrix values.
4. Calculate recall for class 1 using the matrix values.
5. Why might accuracy alone be insufficient when classes are imbalanced?
6. Which metric would you pay particular attention to if missing a positive case is very costly? Explain.

## 9. Challenge – Lab 4: Compare Thresholds
Try thresholds of `0.30`, `0.50`, and `0.70` for class 1. Compare the confusion matrices and classification reports.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

probabilities = model.predict_proba(X_test_scaled)[:, 1]

for threshold in [0.30, 0.50, 0.70]:
    custom_pred = (probabilities >= threshold).astype(int)

    print("\\nThreshold:", threshold)
    print(confusion_matrix(y_test, custom_pred))
    print(classification_report(
        y_test,
        custom_pred,
        target_names=["malignant", "benign"],
        zero_division=0
    ))

## 10. AI Application – Lab 4

Evaluation is important whenever a model supports a real decision. In a screening-support workflow, a team may focus on reducing missed cases. In a fraud-alert workflow, a team may need to balance missed fraud against the cost of investigating false alerts.

The best metric and threshold depend on the application, the data, and expert review. This notebook's model is for learning only, not for real medical decisions.


---
# Module 4 – Final Mini Project

## Project: Build and Evaluate a Binary Classification Model

### Goal
Use the Breast Cancer practice dataset to complete a small end-to-end machine-learning workflow.

### Tasks
1. Load the dataset and print its shape.
2. Display the number of records in each class.
3. Split the data into training and testing sets using stratification.
4. Scale features using training-set statistics only.
5. Train Logistic Regression.
6. Generate predictions and class-1 probabilities.
7. Display a confusion matrix and classification report.
8. Calculate ROC-AUC.
9. Write a short conclusion describing what the metrics tell you.

### Student Conclusion Template
Complete these sentences in your own words:
- My model was trained using __________.
- The test accuracy / evaluation results show __________.
- One type of error made by the model is __________.
- Precision tells me __________.
- Recall tells me __________.
- One limitation of this experiment is __________.

### Submission Checklist
- [ ] All four labs run without errors.
- [ ] Exercises are completed.
- [ ] Challenge tasks are attempted.
- [ ] Confusion matrix is interpreted correctly.
- [ ] Positive class is clearly identified.
- [ ] Final conclusion is written.

## Final Reminder
The goal is not only to run code. You should be able to explain what each step does, why it is needed, and how to interpret the results.
